# Phase 2 — OCR and rule-based baseline

This notebook runs Tesseract on the prepared SROIE receipt images, extracts words and boxes, applies a simple image-preprocessing step, and evaluates a transparent heuristic baseline against the held-out SROIE test annotations. The baseline is only a reference point; it is not a trained model.

## Colab setup
Open the Phase 1 notebook's Colab project or upload this notebook into Colab. Run cells in order. Use the same Google Drive account and project folder created in Phase 1. OCR runs on CPU and can take several minutes across the test split. The notebook saves a per-document cache so an interrupted run can resume.

Tesseract is a separate system OCR engine; `pytesseract` is its Python wrapper. This notebook uses `image_to_data` to retain word boxes and confidence values. [Tesseract install instructions](https://github.com/tesseract-ocr/tessdoc/blob/main/Installation.md), [pytesseract implementation/API](https://github.com/madmaze/pytesseract/blob/master/pytesseract/pytesseract.py).

## Cell 1 — Install Tesseract and Python packages
Colab's Ubuntu runtime installs the OCR executable with apt; pip installs the Python wrapper and image/data libraries.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr
%pip -q install pytesseract pillow pandas matplotlib tqdm

## Cell 2 — Mount Drive and check Tesseract

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import pytesseract
print("Tesseract:", pytesseract.get_tesseract_version())

## Cell 3 — Set project paths and load the Phase 1 test records
This reads the untouched official test split created in Phase 1. If the file or any copied image is missing, return to Phase 1 and complete its image recovery and output validation first.

In [ ]:
from pathlib import Path
import json

DRIVE_ROOT = Path("/content/drive/MyDrive/intelligent-document-ai")
DATA_PROCESSED = DRIVE_ROOT / "data" / "processed"
OUTPUTS = DRIVE_ROOT / "outputs"
METRICS = DRIVE_ROOT / "metrics"
OUTPUTS.mkdir(parents=True, exist_ok=True)
METRICS.mkdir(parents=True, exist_ok=True)

TEST_JSONL = DATA_PROCESSED / "test.jsonl"
if not TEST_JSONL.is_file():
    raise FileNotFoundError(f"Missing {TEST_JSONL}. Complete Phase 1 first.")

test_records = []
with TEST_JSONL.open(encoding="utf-8") as file:
    for line in file:
        if line.strip():
            row = json.loads(line)
            if not Path(row["image_path"]).is_file():
                raise FileNotFoundError(f"Missing image: {row['image_path']}")
            test_records.append(row)
print(f"Loaded {len(test_records)} official test documents.")

## Cell 4 — Define image preprocessing and OCR
The preprocessing is intentionally modest: convert to grayscale, improve contrast, and upscale small images. Tesseract returns word-level pixel boxes; we convert those boxes to LayoutLM-compatible 0–1000 coordinates while retaining raw pixels and OCR confidence for display/evaluation.

In [ ]:
from PIL import Image, ImageOps, ImageEnhance
import pytesseract
from pytesseract import Output

def preprocess_receipt(image):
    gray = ImageOps.grayscale(image)
    gray = ImageOps.autocontrast(gray)
    # Small receipt scans can benefit from a modest upscale.
    if max(gray.size) < 1800:
        scale = 1800 / max(gray.size)
        gray = gray.resize((round(gray.width * scale), round(gray.height * scale)))
    return ImageEnhance.Contrast(gray).enhance(1.15)

def normalize_xyxy(box, image_size):
    width, height = image_size
    x0, y0, x1, y1 = box
    return [
        max(0, min(1000, round(1000 * x0 / width))),
        max(0, min(1000, round(1000 * y0 / height))),
        max(0, min(1000, round(1000 * x1 / width))),
        max(0, min(1000, round(1000 * y1 / height))),
    ]

def run_ocr(image_path):
    original = Image.open(image_path).convert("RGB")
    processed = preprocess_receipt(original)
    data = pytesseract.image_to_data(
        processed, lang="eng", config="--oem 3 --psm 6", output_type=Output.DICT
    )
    scale_x = original.width / processed.width
    scale_y = original.height / processed.height
    tokens = []
    for i, text in enumerate(data["text"]):
        text = text.strip()
        try:
            confidence = float(data["conf"][i])
        except (TypeError, ValueError):
            confidence = -1.0
        if not text or confidence < 0:
            continue
        left = round(data["left"][i] * scale_x)
        top = round(data["top"][i] * scale_y)
        right = round((data["left"][i] + data["width"][i]) * scale_x)
        bottom = round((data["top"][i] + data["height"][i]) * scale_y)
        pixel_box = [left, top, right, bottom]
        tokens.append({
            "text": text,
            "confidence": confidence,
            "bbox": normalize_xyxy(pixel_box, original.size),
            "pixel_bbox": pixel_box,
            "line_id": (data["block_num"][i], data["par_num"][i], data["line_num"][i]),
        })
    return original, tokens

def group_tokens_into_lines(tokens):
    grouped = {}
    for token in tokens:
        grouped.setdefault(token["line_id"], []).append(token)
    lines = []
    for line_tokens in grouped.values():
        line_tokens.sort(key=lambda item: item["pixel_bbox"][0])
        lines.append({
            "text": " ".join(item["text"] for item in line_tokens),
            "bbox": [
                min(item["bbox"][0] for item in line_tokens),
                min(item["bbox"][1] for item in line_tokens),
                max(item["bbox"][2] for item in line_tokens),
                max(item["bbox"][3] for item in line_tokens),
            ],
            "confidence": sum(item["confidence"] for item in line_tokens) / len(line_tokens),
        })
    return sorted(lines, key=lambda line: (line["bbox"][1], line["bbox"][0]))

## Cell 5 — Run OCR on one example and visualize word boxes
Inspect the OCR text and red boxes before running the full test set. If boxes look misplaced, check the source image dimensions and preprocessing scale.

In [ ]:
import matplotlib.pyplot as plt
from PIL import ImageDraw

preview_image, preview_tokens = run_ocr(test_records[0]["image_path"])
preview_overlay = preview_image.copy()
draw = ImageDraw.Draw(preview_overlay)
for token in preview_tokens:
    draw.rectangle(token["pixel_bbox"], outline="red", width=2)
print("OCR word count:", len(preview_tokens))
print("OCR preview:", " ".join(token["text"] for token in preview_tokens[:80]))
plt.figure(figsize=(10, 12))
plt.imshow(preview_overlay)
plt.axis("off");

## Cell 6 — Rule-based field extraction
Heuristics use OCR text and layout only. Company is approximated by the first plausible top line; date uses common numeric/month patterns; address uses address words/postcodes; total uses amount-looking text and prefers lines containing total-related cues. This simple baseline will miss variation and OCR errors.

In [ ]:
import re

FIELDS = ("COMPANY", "DATE", "ADDRESS", "TOTAL")
DATE_PATTERN = re.compile(
    r"\b(?:\d{1,2}[./-]\d{1,2}[./-]\d{2,4}|\d{4}[./-]\d{1,2}[./-]\d{1,2}|"
    r"\d{1,2}\s+(?:JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG|SEP|OCT|NOV|DEC)[A-Z]*\s+\d{2,4})\b",
    re.IGNORECASE,
)
AMOUNT_PATTERN = re.compile(
    r"(?<![A-Z0-9])(?:RM|MYR|USD|\$|£|€)?\s*-?\d{1,3}(?:,\d{3})*(?:\.\d{2})(?!\d)",
    re.IGNORECASE,
)
ADDRESS_HINTS = ("JALAN", "JLN", "STREET", "ROAD", "RD.", "AVENUE", "AVE", "LORONG", "TAMAN", "SUITE", "NO.")
TOTAL_HINTS = ("GRAND TOTAL", "TOTAL", "AMOUNT DUE", "NET", "BALANCE")

def first_match(pattern, text):
    match = pattern.search(text)
    return match.group(0).strip() if match else ""

def extract_baseline_fields(tokens):
    lines = group_tokens_into_lines(tokens)
    result = {field: "" for field in FIELDS}
    if not lines:
        return result

    # Company: first plausible text line in the top 30% of the receipt.
    top_lines = [line for line in lines if line["bbox"][1] <= 300]
    excluded = ("RECEIPT", "INVOICE", "TAX INVOICE", "CASH BILL")
    for line in top_lines:
        text = line["text"].strip()
        letters = sum(char.isalpha() for char in text)
        if letters >= 3 and not any(cue in text.upper() for cue in excluded):
            result["COMPANY"] = text
            break

    # Date: first line with a supported date pattern.
    for line in lines:
        value = first_match(DATE_PATTERN, line["text"])
        if value:
            result["DATE"] = value
            break

    # Address: prioritize address cues; use a postcode-like pattern as a fallback cue.
    candidates = []
    for line in lines:
        text = line["text"].strip()
        upper = text.upper()
        has_hint = any(hint in upper for hint in ADDRESS_HINTS)
        has_postcode = bool(re.search(r"\b\d{5}\b", text))
        has_letters_and_digits = any(c.isalpha() for c in text) and any(c.isdigit() for c in text)
        if line["bbox"][1] < 650 and has_letters_and_digits and (has_hint or has_postcode):
            candidates.append((int(has_hint) + int(has_postcode), len(text), text))
    if candidates:
        result["ADDRESS"] = max(candidates)[2]

    # Total: choose a currency/decimal amount on a total-cue line when possible.
    amount_candidates = []
    for line in lines:
        for match in AMOUNT_PATTERN.finditer(line["text"]):
            cue_score = int(any(cue in line["text"].upper() for cue in TOTAL_HINTS))
            amount_candidates.append((cue_score, line["bbox"][1], match.group(0).strip()))
    if amount_candidates:
        result["TOTAL"] = sorted(amount_candidates, key=lambda item: (item[0], item[1]))[-1][2]
    return result

def reconstruct_gold_fields(words, tags):
    spans = {field: [] for field in FIELDS}
    current_field, current_words = None, []
    def finish():
        if current_field and current_words:
            spans[current_field].append(" ".join(current_words))
    for word, tag in zip(words, tags):
        if tag == "O":
            finish()
            current_field, current_words = None, []
        elif tag.startswith("B-"):
            finish()
            current_field = tag[2:]
            current_words = [word]
        elif tag.startswith("I-") and current_field == tag[2:]:
            current_words.append(word)
        else:
            finish()
            current_field, current_words = None, []
    finish()
    return {field: " ".join(spans[field]).strip() for field in FIELDS}

def normalize_for_match(value):
    return re.sub(r"[^a-z0-9]+", "", str(value).lower())

## Cell 7 — Evaluate on official test and save OCR predictions
The cache stores one JSONL record per completed document. If Colab disconnects, rerunning this cell skips IDs already in the cache. Set `RESET_OCR_CACHE = True` to deliberately start over. The saved OCR token records include text, normalized boxes, pixel boxes, and confidence.

In [ ]:
from tqdm.auto import tqdm
from collections import Counter

OCR_CACHE = OUTPUTS / "ocr_test_predictions.jsonl"
RESET_OCR_CACHE = False
if RESET_OCR_CACHE and OCR_CACHE.exists():
    OCR_CACHE.unlink()

cached = {}
if OCR_CACHE.exists():
    with OCR_CACHE.open(encoding="utf-8") as file:
        for line in file:
            if line.strip():
                row = json.loads(line)
                cached[row["document_id"]] = row

for record in tqdm(test_records, desc="Tesseract OCR"):
    doc_id = str(record["document_id"])
    if doc_id in cached:
        continue
    image, tokens = run_ocr(record["image_path"])
    prediction = extract_baseline_fields(tokens)
    gold = reconstruct_gold_fields(record["words"], record["ner_tags"])
    output = {
        "document_id": doc_id,
        "image_path": record["image_path"],
        "ocr_tokens": tokens,
        "prediction": prediction,
        "gold": gold,
    }
    with OCR_CACHE.open("a", encoding="utf-8") as file:
        file.write(json.dumps(output, ensure_ascii=False) + "\n")
    cached[doc_id] = output

# Keep only current test IDs and ensure each row is represented.
test_ids = {str(row["document_id"]) for row in test_records}
results = [cached[doc_id] for doc_id in test_ids]
if len(results) != len(test_records):
    raise ValueError(f"Expected {len(test_records)} cached test results, found {len(results)}")
print("OCR results ready:", len(results))

## Cell 8 — Calculate baseline field metrics and save report
Exact match compares normalized predicted and gold field strings. A field counts as correct only when the full value matches. These metrics are meaningful as a simple baseline but are sensitive to OCR transcription and the normalization rule; review errors before drawing conclusions.

In [ ]:
import pandas as pd

counts = {field: {"correct": 0, "eligible": 0, "tp": 0, "fp": 0, "fn": 0}
          for field in FIELDS}
for row in results:
    for field in FIELDS:
        pred = normalize_for_match(row["prediction"].get(field, ""))
        gold = normalize_for_match(row["gold"].get(field, ""))
        if gold:
            counts[field]["eligible"] += 1
            if pred == gold:
                counts[field]["correct"] += 1
                counts[field]["tp"] += 1
            else:
                counts[field]["fn"] += 1
                if pred:
                    counts[field]["fp"] += 1
        elif pred:
            counts[field]["fp"] += 1

metric_rows = []
for field, values in counts.items():
    precision = values["tp"] / (values["tp"] + values["fp"]) if values["tp"] + values["fp"] else 0.0
    recall = values["tp"] / (values["tp"] + values["fn"]) if values["tp"] + values["fn"] else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    exact = values["correct"] / values["eligible"] if values["eligible"] else 0.0
    metric_rows.append({
        "field": field, "exact_match_accuracy": exact,
        "precision": precision, "recall": recall, "f1": f1,
        "correct": values["correct"], "gold_present": values["eligible"],
    })

metrics_df = pd.DataFrame(metric_rows)
display(metrics_df)
report = {
    "dataset": "darentang/sroie",
    "evaluated_split": "official test",
    "documents": len(results),
    "match_rule": "case-insensitive alphanumeric-only full-field exact match",
    "metrics_by_field": metric_rows,
}
(METRICS / "ocr_baseline_metrics.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
pd.DataFrame([
    {"document_id": row["document_id"], **{f"pred_{k.lower()}": row["prediction"].get(k, "")
      for k in FIELDS}, **{f"gold_{k.lower()}": row["gold"].get(k, "") for k in FIELDS}}
    for row in results
]).to_csv(OUTPUTS / "ocr_baseline_test_predictions.csv", index=False)
print("Metrics saved:", METRICS / "ocr_baseline_metrics.json")
print("Field predictions saved:", OUTPUTS / "ocr_baseline_test_predictions.csv")
print("Full word-level OCR cache saved:", OCR_CACHE)

## Expected outputs, limits, and recovery
Drive outputs:
- `outputs/ocr_test_predictions.jsonl`: OCR tokens, boxes, confidence, predictions, and gold fields per test receipt.
- `outputs/ocr_baseline_test_predictions.csv`: extracted fields beside annotations.
- `metrics/ocr_baseline_metrics.json`: field exact-match accuracy, precision, recall, and F1.

If interrupted, rerun setup and mount Drive, then rerun from Cell 3. The cache skips completed receipt IDs. This baseline uses hand-written patterns and position cues; it can miss fields or produce incorrect values. Tesseract quality depends on image quality, language data, and preprocessing. Do not interpret its results as LayoutLMv3 performance or as support for invoices/bank statements.